here only plot the precomputed results for the resnets. reeeally simply explain model and data + training. 

> explain random dir generation

> naive approach plots look degenerate [noskip vs skip]

> final approach with filternorm [noskip vs skip]

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from loss_landscapes.paths import PROJ_ROOT
from loss_landscapes.plotly.templates import use
use("mine_dark")

ZMAX_LOSS = 30      # clip the loss here before taking the log
EPS = 1e-6          # avoid log(0)
Z_PAD = 0.02        # extra z-axis range (fraction of the span) so clipped surfaces don't z-fight with the box

# load all results, keyed by (depth, method, skip) parsed from "resnet_<depth>_<skip>_<method>.npz"
results = {}
for path in sorted((PROJ_ROOT / "results").glob("resnet_*.npz")):
    _, depth, skip, method = path.stem.split("_")
    res = np.load(path)
    results[(depth, method, skip)] = dict(
        alphas=res["alphas"],
        betas=res["betas"],
        z=np.log(np.clip(res["z"], EPS, ZMAX_LOSS)),
    )

# shared z range over ALL results (both methods), so every panel in every figure is comparable
zrange = [min(r["z"].min() for r in results.values()), np.log(ZMAX_LOSS)]


def plot_landscapes(method: str):
    """ grid of surfaces for one direction method: one row per depth, columns = no skip | skip """
    depths = sorted({d for d, m, _ in results if m == method})
    depths = [d for d in depths if (d, method, "noskip") in results and (d, method, "skip") in results]

    fig = make_subplots(
        rows=len(depths), cols=2,
        specs=[[{"type": "surface"}, {"type": "surface"}] for _ in depths],
        subplot_titles=[f"{d} | {s}" for d in depths for s in ("no skip", "skip")],
        horizontal_spacing=0.02,
        vertical_spacing=0.05,
    )

    for i, depth in enumerate(depths, start=1):
        for j, skip in enumerate(["noskip", "skip"], start=1):
            r = results[(depth, method, skip)]
            fig.add_trace(
                go.Surface(x=r["alphas"], y=r["betas"], z=r["z"].T, coloraxis="coloraxis"),
                row=i, col=j,
            )

    pad = Z_PAD * (zrange[1] - zrange[0])
    fig.update_scenes(   # applies to every 3D subplot
        zaxis=dict(range=[zrange[0] - pad, zrange[1] + pad]),
        xaxis_title="α", yaxis_title="β", zaxis_title="log loss",
    )
    fig.update_layout(
        title=f"directions: {method}",
        coloraxis=dict(colorscale="dense", cmin=zrange[0], cmax=zrange[1], showscale=False),
        margin=dict(t=60, b=20, l=20, r=20),
        height=450 * len(depths),
    )
    return fig


In [ ]:
plot_landscapes("naive").show()

In [ ]:
plot_landscapes("filternorm").show()